# Prétraitement et Fusion Multi-Omique (v2)

Ce notebook utilise la classe `ClinicalPreprocessor` pour préparer les données pour la modélisation.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import joblib
import importlib

# Robust path setup
project_root = Path(os.getcwd())
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Racine du projet : {project_root}")

# Import common components
from xai_clinical.data.pancan_loader import PANCANLoader, BRCALoader, PANCANBRCAFusion
# Force reload to ensure the new class is picked up
import xai_clinical.data.preprocessor
importlib.reload(xai_clinical.data.preprocessor)
from xai_clinical.data.preprocessor import ClinicalPreprocessor


Racine du projet : c:\Users\nidha\Desktop\xai_clinical_prediction


## 1. Chargement des Données

In [2]:
pancan_dir = project_root / "data" / "raw" / "pancan"
brca_dir = project_root / "data" / "raw" / "brca_tcga"

pancan_loader = PANCANLoader(str(pancan_dir))
brca_loader = BRCALoader(str(brca_dir))

clinical_df = brca_loader.get_merged_clinical()
target = brca_loader.extract_survival_target(clinical_df, cutoff_months=24)
expression_df = pancan_loader.load_gene_expression()

fusion = PANCANBRCAFusion(pancan_loader, brca_loader)
fused_df = fusion.fuse_expression_clinical(expression_df, clinical_df, target)

X = fused_df.drop(columns=['TARGET'])
y = fused_df['TARGET']

print(f"Dataset prêt : X={X.shape}, y={y.value_counts().to_dict()}")

2026-04-29 22:57:30,652 - xai_clinical.data.pancan_loader - INFO - Clinical patient: (1097, 110)
2026-04-29 22:57:30,662 - xai_clinical.data.pancan_loader - INFO - Clinical sample: (1108, 26)
2026-04-29 22:57:30,672 - xai_clinical.data.pancan_loader - INFO - Clinical merged: (1108, 136)
2026-04-29 22:57:30,678 - xai_clinical.data.pancan_loader - INFO - Cible survie calculée (cut-off 24m): {0: 1062, 1: 39}
2026-04-29 22:57:30,681 - xai_clinical.data.pancan_loader - INFO - Chargement PANCAN expression: c:\Users\nidha\Desktop\xai_clinical_prediction\data\raw\pancan\EBPlusPlusAdjustPANCAN.tsv


MemoryError: 

## 2. Prétraitement avec ClinicalPreprocessor

On effectue l'imputation, le scaling et la sélection de features.

In [ ]:
preprocessor = ClinicalPreprocessor(n_features=150)

# Split train/test interne au preprocessor ou manuel
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=42)

X_train_proc, y_train_proc = preprocessor.fit_transform(X_train, y_train, X_val, X_test)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print(f"Entraînement : {X_train_proc.shape}")
print(f"Validation : {X_val_proc.shape}")
print(f"Test : {X_test_proc.shape}")

c:\Users\nidha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [    5  3526  3527  4370  4371  4376  4654  4655  4808  4809  4815  4817
  4819  4824  4829  5288  6029  6075  7661  7662  7663  7664  7665  9297
  9306  9309  9311  9313  9318  9320  9452 10121 12559 13520 13992 14159
 14160 14162 14757 14761 15139 15141 15142 15448 16567 16569 16570 16572
 16573 16575 16579 16580 16606 16635 16639 16679 16698 16699 16700 16701
 16702 16704 16705 16706 16707 16708 16709 16710 16711 16712 16713 16714
 16715 16716 16717 16718 16719 16720 16721 16722 16723 16724 16725 16726
 16727 16728 16729 16730 16731 16732 16733 16734 16735 16736 16737 16738
 16739 16740 16741 16742 16743 16744 16745 16746 16747 16749 16750 16751
 16752 16753 16754 16755 16756 16758 16759 16760 16761 16762 16763 16764
 16765 16766 16767 16768 16769 16770 16771 16772 16773 16775 16776 16777
 16778 16779 16780 16781 16782 16783 16784 167

Entraînement : (745, 150)
Validation : (132, 150)
Test : (220, 150)


c:\Users\nidha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 3. Sauvegarde des Données Préparées

In [ ]:
processed_dir = project_root / "data" / "processed"
os.makedirs(processed_dir, exist_ok=True)

data_to_save = {
    'X_train': X_train_proc,
    'y_train': y_train_proc,
    'X_val': X_val_proc,
    'y_val': y_val,
    'X_test': X_test_proc,
    'y_test': y_test,
    'feature_names': preprocessor.selected_features
}

joblib.dump(data_to_save, processed_dir / "processed_data_v2.pkl")
joblib.dump(preprocessor, processed_dir / "preprocessor_v2.pkl")

print("Données et préprocesseur sauvegardés.")

Données et préprocesseur sauvegardés.
